# Day 27: Agent Protocols and Advanced Use Cases
**MCP, A2A, agentic handoff and a data analysis agent, with a senior FDE session**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup
We pin the MCP Python SDK below version 2 because v2 renamed FastMCP. The rest works offline.

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai scikit-learn numpy pandas "mcp<2" nest_asyncio fastapi httpx

# ===== Option B: VS Code / local =====
# pip install openai scikit-learn numpy pandas "mcp<2" nest_asyncio fastapi httpx
# Create a .env file or set the variable in your terminal:  OPENAI_API_KEY=sk-...
# No key? Everything still runs in OFFLINE demo mode with a tiny built-in stand-in model.
print("Setup done")

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
"""
import json, os, re
import numpy as np

def _get_key():
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")                 # change to any current chat model you have access to
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None):
    if ONLINE:
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}])
        return r.choices[0].message.content
    return offline_llm(prompt, system)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, ONLINE

In [ ]:
import asyncio, json, sys

def run_async(coro):
    """Run async code in scripts and in notebooks (which already have an event loop)."""
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    import nest_asyncio
    nest_asyncio.apply()
    return loop.run_until_complete(coro)
print("ready")

## 2. An MCP server

In [ ]:
%%writefile mcp_server.py
import json, re
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("orders")
ORDERS = {"ORD-1042": {"status": "delivered", "amount": 4800, "days_late": 3},
          "ORD-2077": {"status": "in transit", "amount": 1200, "days_late": 0}}

@mcp.tool()
def get_order_status(order_id: str) -> str:
    """Look up an order by id, for example ORD-1042. Returns status, amount and days late."""
    if not re.fullmatch(r"ORD-\d{4}", order_id):
        return json.dumps({"error": "order id must look like ORD-1234"})
    return json.dumps(ORDERS.get(order_id, {"error": "not found"}))

@mcp.tool()
def calculate_refund(amount: float, days_late: int) -> str:
    """Late delivery refund: 5% of the amount per day late, capped at 50%."""
    return json.dumps({"refund": round(amount * min(0.05 * days_late, 0.5), 2)})

@mcp.resource("policy://refunds")
def refund_policy() -> str:
    """The refund policy text."""
    return "Late deliveries earn 5% of the order value per day late, up to 50%."

@mcp.prompt()
def apology_email(customer: str, issue: str) -> str:
    """A template for a short apology email."""
    return f"Write a short, warm apology email to {customer} about: {issue}. Offer one concrete next step."

if __name__ == "__main__":
    mcp.run()

## 3. An MCP client: discover and call tools over stdio

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def explore():
    params = StdioServerParameters(command=sys.executable, args=["mcp_server.py"])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print("tools:", [t.name for t in tools.tools])
            resources = await session.list_resources()
            print("resources:", [str(r.uri) for r in resources.resources])
            prompts = await session.list_prompts()
            print("prompts:", [p.name for p in prompts.prompts])
            status = await session.call_tool("get_order_status", {"order_id": "ORD-1042"})
            order = json.loads(status.content[0].text)
            print("order:", order)
            refund = await session.call_tool("calculate_refund", {"amount": order["amount"], "days_late": order["days_late"]})
            print("refund:", refund.content[0].text)
            bad = await session.call_tool("get_order_status", {"order_id": "DROP TABLE"})
            print("invalid input:", bad.content[0].text)
            policy = await session.read_resource("policy://refunds")
            print("policy:", policy.contents[0].text)

run_async(explore())

### TODO 1: Add a `search_policy(query)` tool to the server and call it from the client.

In [ ]:
# TODO: write your code here

## 4. A2A-style delegation (simplified)
Two small FastAPI agents. The refund agent publishes an **agent card** and accepts **tasks**; the support agent discovers it and delegates. This follows A2A's ideas in simplified form; real A2A uses a richer specification.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel
import uuid

refund_agent = FastAPI(title="Refund Agent")
CARD = {"name": "Refund Agent", "description": "Calculates late-delivery refunds.", "url": "http://refund-agent.local",
        "version": "1.0", "skills": [{"id": "calculate_refund", "description": "Refund for an amount and days late"}]}

class TaskRequest(BaseModel):
    skill: str
    message: dict

@refund_agent.get("/.well-known/agent.json")
def agent_card():
    return CARD

@refund_agent.post("/tasks")
def create_task(req: TaskRequest):
    task_id = str(uuid.uuid4())[:8]
    if req.skill != "calculate_refund":
        return {"id": task_id, "status": "failed", "error": f"unknown skill {req.skill}"}
    m = req.message
    refund = round(m["amount"] * min(0.05 * m["days_late"], 0.5), 2)
    return {"id": task_id, "status": "completed", "artifacts": [{"type": "data", "data": {"refund": refund}}]}

remote = TestClient(refund_agent)

def support_agent(question, order):
    card = remote.get("/.well-known/agent.json").json()
    skills = [s["id"] for s in card["skills"]]
    print("discovered:", card["name"], "skills:", skills)
    if "calculate_refund" not in skills:
        return "No agent can calculate refunds."
    task = remote.post("/tasks", json={"skill": "calculate_refund", "message": {"amount": order["amount"], "days_late": order["days_late"]}}).json()
    print("task:", task["id"], task["status"])
    refund = task["artifacts"][0]["data"]["refund"]
    return f"Thanks for waiting. Your refund for the late delivery is {refund} INR."

print(support_agent("What refund do I get?", {"amount": 4800, "days_late": 3}))

## 5. Agentic handoff

In [ ]:
from typing import Literal

class Handoff(BaseModel):
    from_agent: str
    to: str
    summary: str
    facts: dict
    steps_tried: list[str]
    reason: str
    suggested_next_step: str
    urgency: Literal["low", "medium", "high"]

h = Handoff(
    from_agent="support-bot", to="human-refunds-team",
    summary="Customer Mei wants a refund for ORD-3310, delivered 9 days late; she is upset.",
    facts={"order_id": "ORD-3310", "amount": 7100, "days_late": 9, "calculated_refund": 3195.0},
    steps_tried=["get_order_status -> delivered 9 days late", "calculate_refund -> 3195.0"],
    reason="Refund above the bot's 2,000 INR approval limit",
    suggested_next_step="Approve 3,195 INR refund and send the apology template",
    urgency="high",
)
print(h.model_dump_json(indent=2))

## 6. A data analysis agent with a code safety check

In [ ]:
import ast, pandas as pd, numpy as np
rng = np.random.default_rng(27)
sales = pd.DataFrame({
    "month": np.repeat(["Sep", "Oct", "Nov"], 4),
    "city": ["Tokyo", "Osaka", "Bengaluru", "Pune"] * 3,
    "revenue": rng.integers(80, 200, 12) * 1000,
    "orders": rng.integers(300, 900, 12),
})

BANNED_NAMES = {"open", "exec", "eval", "compile", "__import__", "globals", "locals", "getattr", "setattr", "os", "sys", "subprocess", "input"}

def check_code(code):
    tree = ast.parse(code)
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            return "imports are not allowed"
        if isinstance(node, ast.Name) and node.id in BANNED_NAMES:
            return f"name '{node.id}' is not allowed"
        if isinstance(node, ast.Attribute) and node.attr.startswith("_"):
            return "private attributes are not allowed"
    return None

def run_sandboxed(code, df):
    problem = check_code(code)
    if problem:
        return f"BLOCKED: {problem}"
    env = {"df": df.copy(), "pd": pd, "__builtins__": {"len": len, "round": round, "sum": sum, "min": min, "max": max}}
    exec(code, env)                       # demo only: production uses a container or managed code interpreter
    return env.get("result")

TEMPLATES = {   # offline stand-in for the model writing pandas code
    "highest revenue city": "result = df.groupby('city')['revenue'].sum().idxmax()",
    "revenue by month": "result = df.groupby('month', sort=False)['revenue'].sum()",
    "average order value": "result = round(df['revenue'].sum() / df['orders'].sum(), 2)",
}

def write_code(question):
    if ONLINE:
        r = llm(f"Write pandas code using an existing DataFrame df with columns {list(sales.columns)}. "
                f"Store the answer in a variable named result. No imports. Code only.\nQuestion: {question}", temperature=0)
        return r.replace("```python", "").replace("```", "").strip()
    return next((c for k, c in TEMPLATES.items() if k in question.lower()), "result = df.head()")

for q in ["Which is the highest revenue city?", "Show revenue by month", "What is the average order value?"]:
    code = write_code(q)
    print(f"Q: {q}\n   code: {code}\n   result: {run_sandboxed(code, sales)}\n")

for evil in ["import os\nresult = os.listdir('.')", "result = open('secrets.txt').read()", "result = df.__class__.__bases__"]:
    print(repr(evil[:40]), "->", run_sandboxed(evil, sales))

### TODO 2: Add a row limit (refuse DataFrames over 100,000 rows) and a check that blocks `while` loops.

In [ ]:
# TODO: write your code here

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)